In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA ÚNICA — LSTM APENAS
# ══════════════════════════════════════════════════════════

# ── IMPORTS ──
!pip install -q wfdb scikit-learn
import os, re, gc, warnings, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt, iirnotch
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    confusion_matrix, roc_auc_score
)
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

# ── CONFIGURAÇÕES ──
BASE_DIR   = '/home/jovyan/work/Untitled Folder/Teste_1 - CNN/Dados'
SESSOES    = ['S1', 'S2', 'S3']
CANAIS_IDX = list(range(16))
N_CANAIS   = len(CANAIS_IDX)
FS         = 2048
WIN_SIZE   = 512
WIN_STEP   = 512
N_CLASSES  = 16
N_FOLDS    = 3   # ← 3 folds
EPOCHS     = 10  # ← 10 épocas
BATCH_SIZE = 32
LR         = 1e-3

# ── FUNÇÕES ──
def extrair_label(nome):
    m = re.search(r'gesture(\d+)', nome, re.IGNORECASE)
    return int(m.group(1)) - 1 if m else None

def filtrar_canal(sinal_1d, fs):
    if len(sinal_1d) < 4 * fs:
        return sinal_1d
    nyq  = fs / 2.0
    low  = 20.0 / nyq
    high = min(450.0, nyq * 0.95) / nyq
    if 0.01 < low < high < 0.99:
        b, a = butter(4, [low, high], btype='band')
        sinal_1d = filtfilt(b, a, sinal_1d)
    w0 = 60.0 / nyq
    if 0.01 < w0 < 0.99:
        b_n, a_n = iirnotch(w0, Q=30)
        sinal_1d = filtfilt(b_n, a_n, sinal_1d)
    return sinal_1d

def carregar_sessao(data_dir):
    arquivos_hea = sorted([f for f in os.listdir(data_dir) if f.endswith('.hea')])
    print(f'  Arquivos .hea: {len(arquivos_hea)}')
    X_list, y_list, erros = [], [], 0
    for i, fname in enumerate(arquivos_hea):
        nome  = os.path.splitext(fname)[0]
        label = extrair_label(nome)
        if label is None or not (0 <= label < N_CLASSES):
            erros += 1; continue
        try:
            rec = wfdb.rdrecord(os.path.join(data_dir, nome))
            if rec.p_signal is None: continue
            sinal = rec.p_signal[:, CANAIS_IDX].astype(np.float32)
            filtrado = np.zeros_like(sinal)
            for c in range(sinal.shape[1]):
                filtrado[:, c] = filtrar_canal(sinal[:, c], int(rec.fs))
            del sinal
            inicio, janelas = 0, []
            while inicio + WIN_SIZE <= filtrado.shape[0]:
                janelas.append(filtrado[inicio:inicio + WIN_SIZE, :])
                inicio += WIN_STEP
            del filtrado
            if janelas:
                X_list.append(np.array(janelas, dtype=np.float16))
                y_list.append(np.full(len(janelas), label, dtype=np.int32))
        except Exception:
            erros += 1
        if (i + 1) % 1000 == 0:
            gc.collect()
            print(f'    {i+1}/{len(arquivos_hea)} processados...')
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    del X_list, y_list
    gc.collect()
    print(f'  X shape: {X.shape} | Erros: {erros}')
    return X, y

def calcular_auc(y_true, y_prob, n_classes):
    cls = np.unique(y_true)
    if len(cls) < 2: return float('nan')
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    try:
        return roc_auc_score(y_bin[:, cls], y_prob[:, cls],
                             multi_class='ovr', average='macro')
    except Exception:
        return float('nan')

def build_lstm(input_shape, n_classes=N_CLASSES, lr=LR):
    inp = layers.Input(shape=input_shape)
    x = layers.LSTM(128, return_sequences=True)(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64,  activation='relu')(x); x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out, name='LSTM')
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('Funcoes definidas.')

# ── CARREGAR DADOS ──
print('\nCarregando todas as sessoes...')
X_parts, y_parts = [], []
for s in SESSOES:
    print(f'\n--- Carregando {s} ---')
    X_s, y_s = carregar_sessao(os.path.join(BASE_DIR, s))
    X_parts.append(X_s)
    y_parts.append(y_s)
    del X_s, y_s
    gc.collect()

X_all = np.concatenate(X_parts, axis=0)
y_all = np.concatenate(y_parts, axis=0)
del X_parts, y_parts
gc.collect()
print(f'\nDataset completo: X={X_all.shape} | y={y_all.shape}')

# ── TREINAMENTO ──
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
results, histories = [], []
all_y_true, all_y_pred = [], []

print(f'\n{"="*60}')
print(f'  MODELO: LSTM')
print(f'{"="*60}')

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
    print(f'\n  ---- FOLD {fold+1}/{N_FOLDS} ----')
    X_tr = X_all[tr_idx].astype(np.float32)
    X_va = X_all[va_idx].astype(np.float32)
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    _, w, ch = X_tr.shape
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr.reshape(-1, ch)).reshape(X_tr.shape)
    X_va = sc.transform(X_va.reshape(-1, ch)).reshape(X_va.shape)

    model = build_lstm((WIN_SIZE, N_CANAIS))
    hist  = model.fit(
        X_tr, to_categorical(y_tr, N_CLASSES),
        validation_data=(X_va, to_categorical(y_va, N_CLASSES)),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=4,
                          restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=2, min_lr=1e-6, verbose=0)
        ], verbose=1
    )
    histories.append(hist)

    y_prob = model.predict(X_va, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_va, y_pred)
    f1  = f1_score(y_va, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_va, y_pred, average='weighted', zero_division=0)
    auc = calcular_auc(y_va, y_prob, N_CLASSES)
    results.append({'fold': fold+1, 'accuracy': acc, 'f1': f1, 'recall': rec, 'auc': auc})
    all_y_true.extend(y_va.tolist())
    all_y_pred.extend(y_pred.tolist())
    print(f'  Acc={acc:.4f} | F1={f1:.4f} | Recall={rec:.4f} | AUC={auc:.4f}')
    del model, X_tr, X_va
    gc.collect()

# ── RESULTADOS ──
df_res = pd.DataFrame(results)
print(f'\n  RESUMO — LSTM')
print(df_res.to_string(index=False))
metricas = {}
for col in ['accuracy', 'f1', 'recall', 'auc']:
    m, s = df_res[col].mean(), df_res[col].std()
    metricas[col] = {'mean': float(m), 'std': float(s)}
    print(f'  {col.upper():10s}: {m:.4f} +/- {s:.4f}')

df_res.to_csv('resultados_LSTM.csv', index=False)
with open('metricas_LSTM.json', 'w') as f:
    _json.dump(metricas, f, indent=2)

# ── CURVAS DE APRENDIZADO ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = plt.cm.tab10(np.linspace(0, 1, N_FOLDS))
for i, h in enumerate(histories):
    axes[0].plot(h.history['accuracy'],     color=cores[i], lw=1.5, label=f'Treino F{i+1}')
    axes[0].plot(h.history['val_accuracy'], color=cores[i], lw=1.5, ls='--', alpha=0.7)
    axes[1].plot(h.history['loss'],         color=cores[i], lw=1.5, label=f'Treino F{i+1}')
    axes[1].plot(h.history['val_loss'],     color=cores[i], lw=1.5, ls='--', alpha=0.7)
axes[0].set_title('Acuracia — LSTM'); axes[0].set_xlabel('Epocas'); axes[0].legend(fontsize=7)
axes[1].set_title('Loss — LSTM');     axes[1].set_xlabel('Epocas'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig('lstm_curvas.png', dpi=150)
plt.show()

# ── MATRIZ DE CONFUSÃO ──
cm      = confusion_matrix(all_y_true, all_y_pred, labels=np.arange(N_CLASSES))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
lbls    = [f'G{i+1:02d}' for i in range(N_CLASSES)]
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues', ax=axes[0], xticklabels=lbls, yticklabels=lbls)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], xticklabels=lbls, yticklabels=lbls)
axes[0].set_title('Confusao LSTM — Contagem');    axes[0].set_xlabel('Predito'); axes[0].set_ylabel('Real')
axes[1].set_title('Confusao LSTM — Normalizada'); axes[1].set_xlabel('Predito'); axes[1].set_ylabel('Real')
plt.tight_layout()
plt.savefig('lstm_confusao.png', dpi=150)
plt.show()

print('\nCONCLUIDO!')

2026-05-01 18:34:07.609083: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 18:34:07.610780: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-01 18:34:07.628566: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-01 18:34:07.628583: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-01 18:34:07.629388: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

TensorFlow: 2.15.0
GPUs: []
Funcoes definidas.

Carregando todas as sessoes...

--- Carregando S1 ---
  Arquivos .hea: 5117


2026-05-01 18:34:08.280215: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-01 18:34:08.283210: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


    1000/5117 processados...
    2000/5117 processados...
    3000/5117 processados...
    4000/5117 processados...
    5000/5117 processados...
  X shape: (96320, 512, 16) | Erros: 301

--- Carregando S2 ---
  Arquivos .hea: 7735
    1000/7735 processados...
    2000/7735 processados...
    3000/7735 processados...
    4000/7735 processados...
    5000/7735 processados...
    7000/7735 processados...
  X shape: (145740, 512, 16) | Erros: 448

--- Carregando S3 ---
  Arquivos .hea: 5117
    1000/5117 processados...
    2000/5117 processados...
    3000/5117 processados...
    4000/5117 processados...
    5000/5117 processados...
  X shape: (96320, 512, 16) | Erros: 301

Dataset completo: X=(338380, 512, 16) | y=(338380,)

  MODELO: LSTM

  ---- FOLD 1/3 ----
Epoch 1/10
7050/7050 [==============================] - 2312s 328ms/step - loss: 2.3086 - accuracy: 0.1808 - val_loss: 1.7219 - val_accuracy: 0.3328 - lr: 0.0010
Epoch 2/10
7050/7050 [==============================] - 2313s 328ms/s